# 06 Music albums — v32 MusicBrainz hard/diverse multihop rebuild

This version uses a MusicBrainz + Wikidata-MBID cache, but fixes the v30/v31 selection problem: output must be 100 records, L5 is required, release-answer tasks avoid over-narrow performer constraints, and L3-L5 emphasize evidence/bridge constraints.

Main policy changes:
- more studio albums than singles (`album=30`, `single=15`);
- performer constraints like ‘albums by The Beatles’ are only allowed for none of the release-answer templates in the selected candidate bank;
- L2–L5 use genre/year plus cross-release evidence conditions instead of direct performer narrowing;
- exact gold is the complete set within the local MusicBrainz-linked cache, with English labels required and RU labels allowed to fall back to original/English titles.


In [1]:

# Load common helpers only if this domain notebook is run standalone.
from pathlib import Path
if "BenchmarkExample" not in globals():
    helper_candidates = [Path("common_helpers.py"), Path("common_helpers(1).py")]
    for hp in helper_candidates:
        if hp.exists():
            exec(hp.read_text(encoding="utf-8"), globals())
            break
    else:
        raise FileNotFoundError("Cannot find common_helpers.py. Run from the project root or place common_helpers.py next to this notebook.")

from dataclasses import asdict
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple, Iterable, Set
from pathlib import Path
import datetime as _dt
import hashlib
import json
import math
import os
import random
import re
import time

import pandas as pd
import requests
from tqdm.auto import tqdm

# -------------------------
# Domain / output config
# -------------------------
MUSIC_DOMAIN_NAME = "music_albums"
MUSIC_GENERATOR_VERSION = "music_releases_performers_v32_musicbrainz_hard_diverse_multihop"

DOMAIN_OUT_DIR = Path(OUT_DIR) / "domain_outputs"
DOMAIN_OUT_DIR.mkdir(parents=True, exist_ok=True)

MUSIC_OUT_PATH = DOMAIN_OUT_DIR / "music_albums.jsonl"
MUSIC_AUDIT_PATH = DOMAIN_OUT_DIR / "music_albums_generation_audit.json"
MUSIC_CHECKPOINT_PATH = DOMAIN_OUT_DIR / "music_albums_generation_checkpoint.json"

MB_CACHE_DIR = Path(OUT_DIR) / "musicbrainz_cache"
MB_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# v32 uses a broader MusicBrainz cache profile to add less-obvious linked release-groups.
# If you want to reuse the already-built v30 cache for quick debugging, set MB_CACHE_VERSION = "v30".
# For final generation, keep v32 so the cache is broader and less biased toward only top artists.
MB_CACHE_VERSION = "v32"
MB_LINKED_RELEASE_GROUPS_PATH = MB_CACHE_DIR / f"linked_release_groups_{MB_CACHE_VERSION}.jsonl"
MB_LINKED_ARTISTS_PATH = MB_CACHE_DIR / f"linked_artists_{MB_CACHE_VERSION}.jsonl"
MB_CACHE_META_PATH = MB_CACHE_DIR / f"musicbrainz_cache_meta_{MB_CACHE_VERSION}.json"

# Do not silently write incomplete datasets like 75/100. If v31 cannot fill all
# L1-L5 targets it raises a clear error with diagnostics instead of overwriting
# the final output with a partial JSONL.
MUSIC_ALLOW_PARTIAL_OUTPUT = False

# If True, rebuild MusicBrainz cache from public API. If False, reuse cached files.
MUSICBRAINZ_FORCE_REBUILD_CACHE = False

# Public MusicBrainz API is rate-limited. Keep this at >= 1.1 unless using a local mirror.
MUSICBRAINZ_MIN_DELAY_SECONDS = 1.15
MUSICBRAINZ_TIMEOUT_SECONDS = 45
MUSICBRAINZ_MAX_RETRIES = 4
MUSICBRAINZ_USER_AGENT = "YandexGPT-reversal-curse-benchmark/0.1 (contact: salam121asd@gmail.com)"

# Query/cache limits. Higher values improve recall but increase first-run cache build time.
MB_SEARCH_PAGE_LIMIT = 100
MB_MAX_TOTAL_PER_SEARCH_QUERY = 400       # paginate search specs up to this many raw MB results
MB_MAX_BROWSE_PER_ARTIST_KIND = 160       # browse artist albums/EPs/singles up to this many rows
MB_MAX_WD_MAPPING_BATCH = 80

# Dataset target.
TARGET_PLAN_MUSIC = {"L1": 10, "L2": 15, "L3": 25, "L4": 25, "L5": 25}
TARGET_TOTAL_MUSIC = sum(TARGET_PLAN_MUSIC.values())

# Answer-kind quotas across the whole domain. "album" bucket means user-facing kind "studio album".
MUSIC_KIND_TARGET_PLAN = {"album": 30, "EP": 25, "single": 15, "performer": 30}

# Per-level targets. These are deterministic and prevent a level from getting stuck on one answer kind.
MUSIC_LEVEL_KIND_PLAN = {
    "L1": {"album": 6, "EP": 4},
    "L2": {"album": 8, "EP": 4, "single": 3},
    "L3": {"album": 7, "EP": 6, "single": 4, "performer": 8},
    "L4": {"album": 6, "EP": 6, "single": 4, "performer": 9},
    "L5": {"album": 3, "EP": 5, "single": 4, "performer": 13},
}

REQUESTED_BY_LEVEL = {"L1": 5, "L2": 5, "L3": 5, "L4": 3, "L5": 3}
MIN_GOLD_BY_LEVEL = {"L1": 5, "L2": 6, "L3": 6, "L4": 5, "L5": 5}
MAX_GOLD_ALLOWED = 80

MUSIC_SEED = 20260524
music_rng = random.Random(MUSIC_SEED)

# Diversity caps. Generation is local, so these caps are cheap and important.
DIVERSITY_CAPS_STRICT = {
    "same_template_per_level": 4,
    "same_artist_total": 2,
    "same_genre_per_level": 4,
    "same_constraint_signature": 1,
}
DIVERSITY_CAPS_RELAXED = {
    "same_template_per_level": 6,
    "same_artist_total": 3,
    "same_genre_per_level": 6,
    "same_constraint_signature": 1,
}

print("output:", MUSIC_OUT_PATH.resolve())
print("audit:", MUSIC_AUDIT_PATH.resolve())
print("cache dir:", MB_CACHE_DIR.resolve())
print("generator:", MUSIC_GENERATOR_VERSION)
print("target plan:", TARGET_PLAN_MUSIC)
print("answer-kind target plan:", MUSIC_KIND_TARGET_PLAN)



✅ Patched: WikidataClient.sparql_select (robust) + load_or_build_pool (safe)
✅ Patched: select_items_with_* используют ru/en fallback + repair_pool_labels чинит QID вместо label
output: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/music_albums.jsonl
audit: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/music_albums_generation_audit.json
cache dir: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/musicbrainz_cache
generator: music_releases_performers_v32_musicbrainz_hard_diverse_multihop
target plan: {'L1': 10, 'L2': 15, 'L3': 25, 'L4': 25, 'L5': 25}
answer-kind target plan: {'album': 30, 'EP': 25, 'single': 15, 'performer': 30}


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## MusicBrainz client and cache helpers

In [2]:

MB_API_BASE = "https://musicbrainz.org/ws/2"


def _json_dump(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def _json_load(path: Path, default: Any = None) -> Any:
    if not path.exists():
        return default
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def _read_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    out = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out


def _write_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    os.replace(tmp, path)


def _sha1_obj(obj: Any) -> str:
    s = json.dumps(obj, ensure_ascii=False, sort_keys=True)
    return hashlib.sha1(s.encode("utf-8")).hexdigest()


class MusicBrainzClient:
    """Small cached client for MusicBrainz public API.

    The public service expects a meaningful User-Agent and no more than ~1 request/sec.
    All responses are cached on disk, so generation itself is local after the first build.
    """
    def __init__(
        self,
        base_url: str = MB_API_BASE,
        cache_dir: Path = MB_CACHE_DIR / "http_cache",
        user_agent: str = MUSICBRAINZ_USER_AGENT,
        min_delay: float = MUSICBRAINZ_MIN_DELAY_SECONDS,
        timeout: int = MUSICBRAINZ_TIMEOUT_SECONDS,
        max_retries: int = MUSICBRAINZ_MAX_RETRIES,
    ):
        self.base_url = base_url.rstrip("/")
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.min_delay = float(min_delay)
        self.timeout = int(timeout)
        self.max_retries = int(max_retries)
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": user_agent, "Accept": "application/json"})
        self._last_ts = 0.0

    def _cache_path(self, endpoint: str, params: Dict[str, Any]) -> Path:
        key = _sha1_obj({"endpoint": endpoint, "params": params})
        return self.cache_dir / f"{key}.json"

    def _sleep_if_needed(self):
        delta = time.time() - self._last_ts
        if delta < self.min_delay:
            time.sleep(self.min_delay - delta)

    def get_json(self, endpoint: str, params: Dict[str, Any], use_cache: bool = True) -> Dict[str, Any]:
        params = dict(params or {})
        params.setdefault("fmt", "json")
        path = self._cache_path(endpoint, params)
        if use_cache and path.exists():
            try:
                return _json_load(path, {})
            except Exception:
                try:
                    path.rename(path.with_suffix(path.suffix + ".bad"))
                except Exception:
                    pass

        url = f"{self.base_url}/{endpoint.lstrip('/')}"
        last_err = None
        for attempt in range(self.max_retries):
            self._sleep_if_needed()
            resp = None
            try:
                resp = self.session.get(url, params=params, timeout=self.timeout)
                self._last_ts = time.time()
                if resp.status_code in (429, 500, 502, 503, 504):
                    ra = resp.headers.get("Retry-After")
                    wait = None
                    if ra:
                        try:
                            wait = float(ra)
                        except Exception:
                            wait = None
                    if wait is None:
                        wait = min(30.0, 2.0 ** attempt) + random.random()
                    last_err = RuntimeError(f"MusicBrainz transient HTTP {resp.status_code}")
                    time.sleep(wait)
                    continue
                resp.raise_for_status()
                data = resp.json()
                if use_cache:
                    _json_dump(path, data)
                return data
            except Exception as e:
                last_err = e
                if attempt == self.max_retries - 1:
                    raise RuntimeError(f"MusicBrainz request failed: endpoint={endpoint} params={params}: {e}") from e
                time.sleep(min(30.0, 2.0 ** attempt) + random.random())
        raise RuntimeError(f"MusicBrainz request failed: {last_err}")

    def search(self, entity_type: str, query: str, limit: int = 100, offset: int = 0) -> Dict[str, Any]:
        return self.get_json(entity_type, {"query": query, "limit": int(limit), "offset": int(offset)})

    def browse_release_groups_by_artist(self, artist_mbid: str, mb_type: str, limit: int = 100, offset: int = 0) -> Dict[str, Any]:
        # Browse API is more reliable than Lucene for curated artist discographies.
        return self.get_json("release-group", {
            "artist": artist_mbid,
            "type": mb_type.lower(),
            "limit": int(limit),
            "offset": int(offset),
        })


mb = MusicBrainzClient()


def _mb_count(data: Dict[str, Any], entity_key: str) -> int:
    for k in ["count", f"{entity_key}-count", f"{entity_key.replace('-', '_')}_count"]:
        if k in data:
            try:
                return int(data[k])
            except Exception:
                pass
    return len(data.get(entity_key, []) or [])


## Curated anchors and MusicBrainz search specs

The cache intentionally uses many artists/genres/year buckets. This gives deterministic diversity and avoids endless random WDQS attempts.

In [3]:

# English names are used for MusicBrainz; Russian strings are only for query NLG.
# Default profile is intentionally bounded: first build should usually take minutes, not hours.
# Switch to "full" if the local candidate bank is too small on your machine.
MUSICBRAINZ_CACHE_BUILD_PROFILE = "broad"  # "fast" | "full" | "broad"

CURATED_ARTIST_NAMES_FAST = [
    "ABBA", "Taylor Swift", "Metallica", "Pink Floyd", "The Beatles", "Madonna", "Queen", "Nirvana",
    "Radiohead", "Beyoncé", "Lady Gaga", "Daft Punk", "Miles Davis", "Bob Dylan", "David Bowie",
    "Michael Jackson", "Rihanna", "The Rolling Stones", "Iron Maiden", "Black Sabbath", "Led Zeppelin",
    "The Cure", "Depeche Mode", "Coldplay", "U2", "Eminem", "Prince", "Kendrick Lamar",
]

CURATED_ARTIST_NAMES_FULL = CURATED_ARTIST_NAMES_FAST + [
    "Adele", "John Coltrane", "Jay-Z", "Björk", "Kanye West", "Kraftwerk", "The Clash",
    "The Smiths", "Sonic Youth", "Blur", "Oasis", "Arctic Monkeys", "The Strokes", "R.E.M.",
]

# Broader/more varied artist seeds: intentionally mixes very famous, mid-tier, cult,
# non-US/UK, jazz, electronic, hip-hop, indie and metal acts. These seeds are only
# for cache discovery; final records are selected with diversity caps and should not
# collapse into "albums by X" prompts.
CURATED_ARTIST_NAMES_BROAD = CURATED_ARTIST_NAMES_FULL + [
    "A Tribe Called Quest", "Portishead", "Massive Attack", "PJ Harvey", "Fela Kuti",
    "Caetano Veloso", "Os Mutantes", "Stereolab", "Yo La Tengo", "The National",
    "Animal Collective", "LCD Soundsystem", "The Flaming Lips", "Pixies", "Fugazi",
    "Pavement", "Wilco", "Sigur Rós", "Mogwai", "Aphex Twin", "Boards of Canada",
    "Burial", "Four Tet", "Nick Cave and the Bad Seeds", "Cocteau Twins", "My Bloody Valentine",
    "Slowdive", "The Replacements", "Talk Talk", "Kate Bush", "Built to Spill",
    "Dinosaur Jr.", "Wire", "Tame Impala", "Grimes", "M83", "Bon Iver", "Fleet Foxes",
    "Mitski", "Sufjan Stevens", "Joanna Newsom", "MF DOOM", "Nas", "Wu-Tang Clan",
    "OutKast", "De La Soul", "J Dilla", "The Roots", "Erykah Badu", "D'Angelo",
    "Janelle Monáe", "Flying Lotus", "Kamasi Washington", "Herbie Hancock", "Charles Mingus",
    "Thelonious Monk", "Alice Coltrane", "Sun Ra", "Nujabes", "Yellow Magic Orchestra",
    "Utada Hikaru", "Fishmans", "X Japan", "BABYMETAL", "Fela Kuti", "Ali Farka Touré",
    "Tinariwen", "Buena Vista Social Club", "Can", "Neu!", "Sparks", "Low", "Big Thief",
]

if MUSICBRAINZ_CACHE_BUILD_PROFILE == "broad":
    CURATED_ARTIST_NAMES = CURATED_ARTIST_NAMES_BROAD
elif MUSICBRAINZ_CACHE_BUILD_PROFILE == "full":
    CURATED_ARTIST_NAMES = CURATED_ARTIST_NAMES_FULL
else:
    CURATED_ARTIST_NAMES = CURATED_ARTIST_NAMES_FAST

GENRE_RU = {
    "rock": "рок-музыка",
    "rock music": "рок-музыка",
    "pop": "поп-музыка",
    "pop music": "поп-музыка",
    "electronic": "электронная музыка",
    "electronic music": "электронная музыка",
    "hip hop": "хип-хоп",
    "hip hop music": "хип-хоп",
    "jazz": "джаз",
    "heavy metal": "хеви-метал",
    "hard rock": "хард-рок",
    "soul": "соул",
    "country": "кантри",
    "punk rock": "панк-рок",
    "alternative rock": "альтернативный рок",
    "folk": "фолк",
    "blues": "блюз",
    "rhythm and blues": "ритм-н-блюз",
    "r&b": "ритм-н-блюз",
    "disco": "диско",
    "reggae": "регги",
    "progressive rock": "прогрессивный рок",
    "house": "хаус",
    "techno": "техно",
    "indie rock": "инди-рок",
    "new wave": "нью-вейв",
    "synth-pop": "синти-поп",
    "funk": "фанк",
    "post-punk": "постпанк",
    "ambient": "эмбиент",
    "experimental": "экспериментальная музыка",
    "shoegaze": "шугейз",
    "industrial": "индастриал",
    "post-rock": "пост-рок",
    "dream pop": "дрим-поп",
    "trip hop": "трип-хоп",
    "drum and bass": "драм-н-бейс",
    "garage rock": "гаражный рок",
    "psychedelic rock": "психоделический рок",
    "art rock": "арт-рок",
    "afrobeat": "афробит",
    "krautrock": "краут-рок",
    "grunge": "гранж",
    "hardcore punk": "хардкор-панк",
    "thrash metal": "трэш-метал",
    "doom metal": "дум-метал",
    "death metal": "дэт-метал",
    "indie pop": "инди-поп",
    "electropop": "электропоп",
}

SEARCH_GENRES_FAST = [
    "rock", "pop", "electronic", "hip hop", "jazz", "heavy metal", "hard rock", "alternative rock",
    "soul", "punk rock", "disco", "reggae", "indie rock", "new wave",
]
SEARCH_GENRES_FULL = SEARCH_GENRES_FAST + ["country", "folk", "blues", "progressive rock", "house", "techno", "synth-pop", "funk"]
SEARCH_GENRES_BROAD = SEARCH_GENRES_FULL + [
    "post-punk", "ambient", "experimental", "shoegaze", "industrial", "noise rock", "post-rock",
    "dream pop", "trip hop", "drum and bass", "garage rock", "psychedelic rock", "art rock",
    "latin", "samba", "bossa nova", "afrobeat", "krautrock", "grunge", "emo", "hardcore punk",
    "thrash metal", "doom metal", "black metal", "death metal", "indie pop", "electropop",
]
if MUSICBRAINZ_CACHE_BUILD_PROFILE == "broad":
    SEARCH_GENRES = SEARCH_GENRES_BROAD
elif MUSICBRAINZ_CACHE_BUILD_PROFILE == "full":
    SEARCH_GENRES = SEARCH_GENRES_FULL
else:
    SEARCH_GENRES = SEARCH_GENRES_FAST

YEAR_BUCKETS_5 = [
    (1960, 1964), (1965, 1969), (1970, 1974), (1975, 1979),
    (1980, 1984), (1985, 1989), (1990, 1994), (1995, 1999),
    (2000, 2004), (2005, 2009), (2010, 2014), (2015, 2019), (2020, 2024),
]
YEAR_BUCKETS_10 = [
    (1960, 1969), (1970, 1979), (1980, 1989), (1990, 1999),
    (2000, 2009), (2010, 2019), (2020, 2024),
]

# For MusicBrainz release-group search. Keep the first-run cache build bounded.
def _mb_date_range(y1: int, y2: int) -> str:
    return f"[{int(y1)}-01-01 TO {int(y2)}-12-31]"


def mb_release_group_search_query(primary_type: str, genre: Optional[str] = None, y1: Optional[int] = None, y2: Optional[int] = None, artist: Optional[str] = None) -> str:
    parts = [f"primarytype:{primary_type}"]
    if genre:
        # MusicBrainz indexed search uses tags/genres as searchable tag-like fields.
        parts.append(f'tag:"{genre}"')
    if artist:
        parts.append(f'artistname:"{artist}"')
    if y1 is not None and y2 is not None:
        parts.append(f"firstreleasedate:{_mb_date_range(y1, y2)}")
    return " AND ".join(parts)


def build_mb_search_specs() -> List[Dict[str, Any]]:
    specs = []
    # Bounded first-run query bank. Artist browse supplies a lot of high-quality discography data;
    # these search specs add genre/year diversity for non-artist and performer-answer tasks.
    if MUSICBRAINZ_CACHE_BUILD_PROFILE == "broad":
        # Wider cache: more genres and year buckets, but still bounded enough for public MB API.
        plan = {
            "EP": {"genres": SEARCH_GENRES[:30], "buckets": YEAR_BUCKETS_5},
            "Single": {"genres": SEARCH_GENRES[:26], "buckets": YEAR_BUCKETS_5},
            "Album": {"genres": SEARCH_GENRES[:30], "buckets": YEAR_BUCKETS_10},
        }
    elif MUSICBRAINZ_CACHE_BUILD_PROFILE == "full":
        plan = {
            "EP": {"genres": SEARCH_GENRES, "buckets": YEAR_BUCKETS_5},
            "Single": {"genres": SEARCH_GENRES, "buckets": YEAR_BUCKETS_5},
            "Album": {"genres": SEARCH_GENRES, "buckets": YEAR_BUCKETS_10},
        }
    else:
        plan = {
            "EP": {"genres": SEARCH_GENRES[:12], "buckets": [(1965, 1969), (1980, 1984), (1990, 1994), (2000, 2004), (2010, 2014), (2015, 2019), (2020, 2024)]},
            "Single": {"genres": SEARCH_GENRES[:10], "buckets": [(1970, 1974), (1980, 1984), (1990, 1994), (2000, 2004), (2010, 2014), (2015, 2019), (2020, 2024)]},
            "Album": {"genres": SEARCH_GENRES[:10], "buckets": [(1960, 1969), (1970, 1979), (1980, 1989), (1990, 1999), (2000, 2009), (2010, 2019), (2020, 2024)]},
        }
    type_to_kind = {"Album": "studio album", "EP": "EP", "Single": "single"}
    for primary_type, cfg in plan.items():
        for genre in cfg["genres"]:
            for y1, y2 in cfg["buckets"]:
                specs.append({
                    "source": "search_release_group",
                    "primary_type": primary_type,
                    "kind": type_to_kind[primary_type],
                    "genre": genre,
                    "year_min": y1,
                    "year_max": y2,
                    "query": mb_release_group_search_query(primary_type, genre=genre, y1=y1, y2=y2),
                })
    return specs

MB_SEARCH_SPECS = build_mb_search_specs()
print("MusicBrainz release-group search specs:", len(MB_SEARCH_SPECS))



MusicBrainz release-group search specs: 938


## Wikidata MBID mapping helpers

In [4]:


def _sparql_string_values(values: Iterable[str]) -> str:
    parts = []
    for v in values:
        s = str(v).replace('\\', '\\\\').replace('"', '\\"')
        parts.append(f'"{s}"')
    return " ".join(parts)


def map_external_ids_to_wikidata(mbids: Iterable[str], prop: str, use_cache: bool = True) -> Dict[str, Dict[str, str]]:
    """Map external IDs (MusicBrainz IDs) to Wikidata items and labels.

    prop=P436 for release-group MBIDs; prop=P434 for artist MBIDs.
    Only rows with English labels are returned, because benchmark gold needs English labels.
    """
    ids = sorted({str(x).strip() for x in mbids if str(x).strip()})
    out: Dict[str, Dict[str, str]] = {}
    if not ids:
        return out
    for i in tqdm(range(0, len(ids), MB_MAX_WD_MAPPING_BATCH), desc=f"WD map {prop}"):
        batch = ids[i:i + MB_MAX_WD_MAPPING_BATCH]
        values = _sparql_string_values(batch)
        sparql = f"""
        SELECT DISTINCT ?item ?mbid ?itemLabelEn ?itemLabelRu WHERE {{
          VALUES ?mbid {{ {values} }}
          ?item wdt:{prop} ?mbid .
          ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .
          OPTIONAL {{ ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") . }}
        }}
        """.strip()
        data = wd.sparql_select(sparql, use_cache=use_cache)
        for r in rows_from_select(data):
            mbid = r.get("mbid")
            qid = uri_to_qid(r.get("item", ""))
            en = r.get("itemLabelEn")
            ru = r.get("itemLabelRu") or en
            if mbid and qid and en and mbid not in out:
                out[mbid] = {"qid": qid, "label_en": en, "label_ru": ru}
    return out


## Build/load MusicBrainz-linked cache

In [5]:

FORBIDDEN_ALBUM_SECONDARY_TYPES = {
    "Compilation", "Live", "Soundtrack", "Remix", "DJ-mix", "Mixtape/Street", "Interview", "Audiobook", "Demo"
}


def norm_text(s: Any) -> str:
    return re.sub(r"\s+", " ", str(s or "").strip()).lower()


def normalize_title_for_dupe(s: str) -> str:
    s = norm_text(s)
    s = re.sub(r"[\W_]+", "", s, flags=re.UNICODE)
    return s


def safe_year_from_date(date_value: Any) -> Optional[int]:
    s = str(date_value or "").strip()
    m = re.match(r"^(\d{4})", s)
    if not m:
        return None
    y = int(m.group(1))
    if 1800 <= y <= 2100:
        return y
    return None


def _artist_credit_to_lists(artist_credit: Any) -> Tuple[List[str], List[str]]:
    mbids, names = [], []
    if not isinstance(artist_credit, list):
        return mbids, names
    for ac in artist_credit:
        art = ac.get("artist") if isinstance(ac, dict) else None
        if not isinstance(art, dict):
            continue
        mbid = art.get("id")
        name = art.get("name") or art.get("sort-name")
        if mbid:
            mbids.append(str(mbid))
        if name:
            names.append(str(name))
    return mbids, names


def _extract_tags(rg: Dict[str, Any]) -> List[str]:
    tags = []
    for key in ["tags", "genres"]:
        vals = rg.get(key) or []
        if isinstance(vals, list):
            for t in vals:
                if isinstance(t, dict):
                    name = t.get("name")
                else:
                    name = str(t)
                if name:
                    tags.append(norm_text(name))
    return sorted(set(tags))


def canonical_kind_from_mb(rg: Dict[str, Any]) -> Optional[str]:
    pt = str(rg.get("primary-type") or rg.get("primary_type") or "").strip()
    secs = set(rg.get("secondary-types") or rg.get("secondary_types") or [])
    if pt == "Album":
        if secs.intersection(FORBIDDEN_ALBUM_SECONDARY_TYPES):
            return None
        return "studio album"
    if pt == "EP":
        return "EP"
    if pt == "Single":
        return "single"
    return None


def release_quota_bucket(kind: str) -> str:
    return "album" if kind == "studio album" else kind


def merge_release_group(raw: Dict[str, Any], spec: Dict[str, Any], store: Dict[str, Dict[str, Any]]):
    mbid = raw.get("id")
    if not mbid:
        return
    title = raw.get("title") or raw.get("name")
    first_date = raw.get("first-release-date") or raw.get("first_release_date") or ""
    first_year = safe_year_from_date(first_date)
    kind = canonical_kind_from_mb(raw)
    if not (title and first_year and kind):
        return
    artist_mbids, artist_names = _artist_credit_to_lists(raw.get("artist-credit") or raw.get("artist_credit"))
    if not artist_mbids:
        return
    row = store.get(mbid)
    if row is None:
        row = {
            "mbid": mbid,
            "title": title,
            "first_release_date": first_date,
            "first_year": first_year,
            "primary_type": raw.get("primary-type"),
            "secondary_types": raw.get("secondary-types") or [],
            "kind": kind,
            "quota_bucket": release_quota_bucket(kind),
            "artist_mbids": artist_mbids,
            "artist_names": artist_names,
            "main_artist_mbid": artist_mbids[0],
            "main_artist_name": artist_names[0] if artist_names else None,
            "tags": _extract_tags(raw),
            "seed_genres": [],
            "source_specs": [],
        }
        store[mbid] = row
    # Merge spec-derived genre even when MusicBrainz did not include tags in result.
    genre = spec.get("genre")
    if genre:
        row.setdefault("seed_genres", [])
        if norm_text(genre) not in row["seed_genres"]:
            row["seed_genres"].append(norm_text(genre))
    row.setdefault("source_specs", [])
    sig = {k: spec.get(k) for k in ["source", "primary_type", "kind", "genre", "year_min", "year_max", "artist_name", "query"] if spec.get(k) is not None}
    if sig not in row["source_specs"]:
        row["source_specs"].append(sig)


def discover_curated_artists() -> List[Dict[str, Any]]:
    """Find MusicBrainz artist MBIDs for curated names, then map them to Wikidata QIDs."""
    found = []
    for name in tqdm(CURATED_ARTIST_NAMES, desc="MB artist search"):
        q = f'artist:"{name}"'
        try:
            data = mb.search("artist", q, limit=5, offset=0)
        except Exception as e:
            print("[WARN] artist search failed", name, e)
            continue
        artists = data.get("artists") or []
        if not artists:
            continue
        # Prefer exact-ish names and actual musical artists/groups.
        def score(a):
            s = int(a.get("score") or 0)
            nm = norm_text(a.get("name"))
            exact = 1000 if nm == norm_text(name) else 0
            typ = 100 if a.get("type") in {"Group", "Person"} else 0
            return exact + typ + s
        a = sorted(artists, key=score, reverse=True)[0]
        if a.get("id"):
            found.append({
                "artist_mbid": a.get("id"),
                "name": a.get("name") or name,
                "sort_name": a.get("sort-name"),
                "type": a.get("type"),
                "seed_name": name,
            })
    amap = map_external_ids_to_wikidata([a["artist_mbid"] for a in found], "P434", use_cache=True)
    out = []
    for a in found:
        wdrow = amap.get(a["artist_mbid"])
        if not wdrow:
            continue
        a = dict(a)
        a.update({"qid": wdrow["qid"], "label_en": wdrow["label_en"], "label_ru": wdrow["label_ru"]})
        out.append(a)
    print("curated artists with WD QID:", len(out), "/", len(CURATED_ARTIST_NAMES))
    return out


def fetch_release_group_search(spec: Dict[str, Any], store: Dict[str, Dict[str, Any]]):
    query = spec["query"]
    first = mb.search("release-group", query, limit=MB_SEARCH_PAGE_LIMIT, offset=0)
    total = _mb_count(first, "release-groups")
    if total <= 0:
        return 0, 0
    if total > MB_MAX_TOTAL_PER_SEARCH_QUERY:
        # Too broad for complete local gold; skip this seed. Narrower specs will cover it.
        return total, 0
    fetched = 0
    pages = [first]
    for off in range(MB_SEARCH_PAGE_LIMIT, total, MB_SEARCH_PAGE_LIMIT):
        pages.append(mb.search("release-group", query, limit=MB_SEARCH_PAGE_LIMIT, offset=off))
    for data in pages:
        for rg in data.get("release-groups") or []:
            merge_release_group(rg, spec, store)
            fetched += 1
    return total, fetched


def fetch_artist_browse_release_groups(artist: Dict[str, Any], mb_type: str, store: Dict[str, Dict[str, Any]]):
    first = mb.browse_release_groups_by_artist(artist["artist_mbid"], mb_type=mb_type, limit=MB_SEARCH_PAGE_LIMIT, offset=0)
    total = _mb_count(first, "release-groups")
    if total <= 0:
        return 0, 0
    limit_total = min(total, MB_MAX_BROWSE_PER_ARTIST_KIND)
    fetched = 0
    pages = [first]
    for off in range(MB_SEARCH_PAGE_LIMIT, limit_total, MB_SEARCH_PAGE_LIMIT):
        pages.append(mb.browse_release_groups_by_artist(artist["artist_mbid"], mb_type=mb_type, limit=MB_SEARCH_PAGE_LIMIT, offset=off))
    for data in pages:
        for rg in data.get("release-groups") or []:
            spec = {"source": "browse_artist_release_group", "artist_name": artist["label_en"], "artist_mbid": artist["artist_mbid"], "primary_type": mb_type.title()}
            merge_release_group(rg, spec, store)
            fetched += 1
    return total, fetched


def build_musicbrainz_linked_cache(force_rebuild: bool = MUSICBRAINZ_FORCE_REBUILD_CACHE) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    if MB_LINKED_RELEASE_GROUPS_PATH.exists() and MB_LINKED_ARTISTS_PATH.exists() and MB_CACHE_META_PATH.exists() and not force_rebuild:
        release_rows = _read_jsonl(MB_LINKED_RELEASE_GROUPS_PATH)
        artist_rows = _read_jsonl(MB_LINKED_ARTISTS_PATH)
        meta = _json_load(MB_CACHE_META_PATH, {})
        print(f"[MB cache] loaded {len(release_rows):,} linked release-groups and {len(artist_rows):,} linked artists")
        return pd.DataFrame(release_rows), pd.DataFrame(artist_rows), meta

    raw_release_groups: Dict[str, Dict[str, Any]] = {}
    stats = {"artist_browse": [], "search": []}

    artists = discover_curated_artists()
    _write_jsonl(MB_LINKED_ARTISTS_PATH.with_suffix(".seed_artists.jsonl"), artists)

    # 1) High-quality artist discography seeds.
    for art in tqdm(artists, desc="MB browse artist release-groups"):
        for mb_type in ["album", "ep", "single"]:
            try:
                total, fetched = fetch_artist_browse_release_groups(art, mb_type, raw_release_groups)
                stats["artist_browse"].append({"artist": art["label_en"], "type": mb_type, "total": total, "fetched": fetched})
            except Exception as e:
                stats["artist_browse"].append({"artist": art.get("label_en"), "type": mb_type, "error": str(e)})

    # 2) Broad genre/year seeds for diversity and performer-answer tasks.
    for spec in tqdm(MB_SEARCH_SPECS, desc="MB search release-groups"):
        try:
            total, fetched = fetch_release_group_search(spec, raw_release_groups)
            stats["search"].append({"query": spec["query"], "kind": spec["kind"], "genre": spec.get("genre"), "year_min": spec.get("year_min"), "year_max": spec.get("year_max"), "total": total, "fetched": fetched})
        except Exception as e:
            stats["search"].append({"query": spec.get("query"), "error": str(e)})

    raw_rows = list(raw_release_groups.values())
    print("raw release-groups collected:", len(raw_rows))

    # Map release group MBIDs to Wikidata QIDs + EN/RU labels.
    rg_map = map_external_ids_to_wikidata([r["mbid"] for r in raw_rows], "P436", use_cache=True)
    artist_mbids = set()
    for r in raw_rows:
        artist_mbids.update(r.get("artist_mbids") or [])
    artist_map = map_external_ids_to_wikidata(artist_mbids, "P434", use_cache=True)

    linked_releases = []
    linked_artists = []

    for mbid, wdrow in artist_map.items():
        linked_artists.append({"artist_mbid": mbid, "qid": wdrow["qid"], "label_en": wdrow["label_en"], "label_ru": wdrow["label_ru"]})

    for r in raw_rows:
        wdrow = rg_map.get(r["mbid"])
        if not wdrow:
            continue
        # Require main artist Wikidata mapping; this keeps performer-answer tasks exact.
        main_artist_mbid = r.get("main_artist_mbid")
        art_wd = artist_map.get(main_artist_mbid) if main_artist_mbid else None
        if not art_wd:
            continue
        all_genres = sorted(set([norm_text(x) for x in (r.get("tags") or []) + (r.get("seed_genres") or []) if x]))
        if not all_genres:
            # Keep artist-based album tasks even without genre, but genre tasks will not use these rows.
            all_genres = []
        row = dict(r)
        row.update({
            "qid": wdrow["qid"],
            "label_en": wdrow["label_en"],
            "label_ru": wdrow["label_ru"],
            "main_artist_qid": art_wd["qid"],
            "main_artist_label_en": art_wd["label_en"],
            "main_artist_label_ru": art_wd["label_ru"],
            "all_genres": all_genres,
        })
        linked_releases.append(row)

    # Deterministic sorting.
    linked_releases = sorted(linked_releases, key=lambda x: (x.get("kind") or "", x.get("first_year") or 0, norm_text(x.get("label_en"))))
    linked_artists = sorted(linked_artists, key=lambda x: norm_text(x.get("label_en")))

    _write_jsonl(MB_LINKED_RELEASE_GROUPS_PATH, linked_releases)
    _write_jsonl(MB_LINKED_ARTISTS_PATH, linked_artists)
    meta = {
        "built_at": utc_now_z(),
        "generator_version": MUSIC_GENERATOR_VERSION,
        "musicbrainz_rate_limit_policy": "Public MusicBrainz API: use meaningful User-Agent and at most about one request per second; responses cached locally.",
        "raw_release_groups_collected": len(raw_rows),
        "linked_release_groups": len(linked_releases),
        "linked_artists": len(linked_artists),
        "stats": stats,
    }
    _json_dump(MB_CACHE_META_PATH, meta)
    print(f"[MB cache] built {len(linked_releases):,} linked release-groups and {len(linked_artists):,} linked artists")
    return pd.DataFrame(linked_releases), pd.DataFrame(linked_artists), meta


release_df, artist_df, mb_cache_meta = build_musicbrainz_linked_cache()
print("release_df:", release_df.shape)
print("artist_df:", artist_df.shape)
if len(release_df):
    print("by kind:", dict(Counter(release_df["kind"])))
    print("year range:", int(release_df["first_year"].min()), int(release_df["first_year"].max()))


MB artist search:   1%|          | 1/115 [00:13<25:17, 13.31s/it]

[WARN] artist search failed ABBA MusicBrainz request failed: endpoint=artist params={'query': 'artist:"ABBA"', 'limit': 5, 'offset': 0, 'fmt': 'json'}: HTTPSConnectionPool(host='musicbrainz.org', port=443): Max retries exceeded with url: /ws/2/artist?query=artist%3A%22ABBA%22&limit=5&offset=0&fmt=json (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:997)')))


MB artist search:   2%|▏         | 2/115 [00:27<25:30, 13.54s/it]

[WARN] artist search failed Taylor Swift MusicBrainz request failed: endpoint=artist params={'query': 'artist:"Taylor Swift"', 'limit': 5, 'offset': 0, 'fmt': 'json'}: HTTPSConnectionPool(host='musicbrainz.org', port=443): Max retries exceeded with url: /ws/2/artist?query=artist%3A%22Taylor+Swift%22&limit=5&offset=0&fmt=json (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:997)')))


MB artist search:   3%|▎         | 3/115 [00:41<25:52, 13.86s/it]

...

[WARN] artist search failed Kraftwerk MusicBrainz request failed: endpoint=artist params={'query': 'artist:"Kraftwerk"', 'limit': 5, 'offset': 0, 'fmt': 'json'}: HTTPSConnectionPool(host='musicbrainz.org', port=443): Max retries exceeded with url: /ws/2/artist?query=artist%3A%22Kraftwerk%22&limit=5&offset=0&fmt=json (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:997)')))


MB artist search:  30%|███       | 35/115 [08:03<18:22, 13.78s/it]

[WARN] artist search failed The Clash MusicBrainz request failed: endpoint=artist params={'query': 'artist:"The Clash"', 'limit': 5, 'offset': 0, 'fmt': 'json'}: HTTPSConnectionPool(host='musicbrainz.org', port=443): Max retries exceeded with url: /ws/2/artist?query=artist%3A%22The+Clash%22&limit=5&offset=0&fmt=json (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:997)')))


MB artist search:  30%|███       | 35/115 [08:05<18:30, 13.88s/it]


KeyboardInterrupt: 

## Local filtering, NLG and record builders

In [ ]:


def listify(x: Any) -> List[Any]:
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        try:
            v = json.loads(x)
            return v if isinstance(v, list) else []
        except Exception:
            return []
    return []


def row_genres(row: Any) -> Set[str]:
    vals = getattr(row, "all_genres", None) if not isinstance(row, dict) else row.get("all_genres")
    return {norm_text(x) for x in listify(vals) if x}


def genre_ru(g: str) -> str:
    return GENRE_RU.get(norm_text(g), str(g))


def kind_ru_acc(kind: str) -> str:
    return {"studio album": "студийных альбомов", "EP": "мини-альбомов (EP)", "single": "синглов", "performer": "исполнителей"}.get(kind, kind)


def kind_en_plural(kind: str) -> str:
    return {"studio album": "studio albums", "EP": "EPs", "single": "singles", "performer": "performers"}.get(kind, kind)


def release_singular_ru(kind: str) -> str:
    return {"studio album": "студийный альбом", "EP": "мини-альбом (EP)", "single": "сингл"}.get(kind, kind)


def release_singular_en(kind: str) -> str:
    return {"studio album": "studio album", "EP": "EP", "single": "single"}.get(kind, kind)


def filter_releases(kind: Optional[str] = None, genre: Optional[str] = None, year_min: Optional[int] = None, year_max: Optional[int] = None, performer: Optional[str] = None, artist_mbid: Optional[str] = None, base_df: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    df = release_df if base_df is None else base_df
    if df is None or len(df) == 0:
        return pd.DataFrame()
    out = df
    if kind:
        out = out[out["kind"] == kind]
    if year_min is not None:
        out = out[out["first_year"].astype(int) >= int(year_min)]
    if year_max is not None:
        out = out[out["first_year"].astype(int) <= int(year_max)]
    if performer:
        n = norm_text(performer)
        out = out[out["main_artist_label_en"].map(norm_text) == n]
    if artist_mbid:
        out = out[out["main_artist_mbid"] == artist_mbid]
    if genre:
        ng = norm_text(genre)
        mask = []
        for row in out.itertuples(index=False):
            mask.append(ng in row_genres(row))
        out = out[mask] if len(mask) else out.iloc[0:0]
    return out.drop_duplicates("qid").copy()


def release_gold_from_df(df: pd.DataFrame) -> List[Dict[str, Any]]:
    if df is None or len(df) == 0:
        return []
    rows = []
    for row in df.sort_values(["label_en", "first_year", "qid"]).itertuples(index=False):
        rows.append({
            "qid": row.qid,
            "label_en": row.label_en,
            "label_ru": row.label_ru or row.label_en,
            "mbid": row.mbid,
            "title": row.label_en,
            "kind": row.kind,
            "year": int(row.first_year),
            "artist_mbid": row.main_artist_mbid,
            "artist_qid": row.main_artist_qid,
            "artist_label_en": row.main_artist_label_en,
            "artist_label_ru": row.main_artist_label_ru,
        })
    return rows


def performer_gold_from_release_df(df: pd.DataFrame) -> List[Dict[str, Any]]:
    if df is None or len(df) == 0:
        return []
    best: Dict[str, Dict[str, Any]] = {}
    for row in df.sort_values(["main_artist_label_en", "first_year", "label_en"]).itertuples(index=False):
        qid = row.main_artist_qid
        if not qid or qid in best:
            continue
        best[qid] = {
            "qid": row.main_artist_qid,
            "label_en": row.main_artist_label_en,
            "label_ru": row.main_artist_label_ru or row.main_artist_label_en,
            "artist_mbid": row.main_artist_mbid,
            "evidence_release_qid": row.qid,
            "evidence_release_mbid": row.mbid,
            "evidence_release_title_en": row.label_en,
            "evidence_release_title_ru": row.label_ru or row.label_en,
            "evidence_release_kind": row.kind,
            "evidence_release_year": int(row.first_year),
        }
    return list(best.values())


def has_bad_public_label_duplicates(gold: List[Dict[str, Any]]) -> bool:
    seen = set()
    for g in gold:
        for key in ["label_en", "label_ru"]:
            n = normalize_title_for_dupe(g.get(key) or "")
            if n:
                tag = (key, n)
                if tag in seen:
                    return True
                seen.add(tag)
    return False


def candidate_ok(gold: List[Dict[str, Any]], level: str, requested: int) -> Tuple[bool, str]:
    n = len(gold)
    if n < MIN_GOLD_BY_LEVEL[level]:
        return False, f"too_few_gold:{n}"
    if n < requested:
        return False, f"less_than_requested:{n}"
    if n > MAX_GOLD_ALLOWED:
        return False, f"too_many_gold:{n}"
    # Higher-level examples should not be all-or-nothing lists of exactly requested_count answers.
    # We prefer a real answer set from which the model can name a requested subset.
    if level in {"L2", "L3", "L4", "L5"} and n <= requested:
        return False, f"gold_too_closed:{n}"
    if has_bad_public_label_duplicates(gold):
        return False, "duplicate_public_labels"
    return True, "ok"


def constraints_signature(c: Dict[str, Any]) -> str:
    return json.dumps(c, ensure_ascii=False, sort_keys=True)


def mb_values_sparql(mbids: List[str]) -> str:
    return _sparql_string_values(mbids)


def exact_sparql_for_gold(answer_kind: str, gold: List[Dict[str, Any]]) -> str:
    if answer_kind == "performer":
        prop = "P434"
        mbids = [g["artist_mbid"] for g in gold]
    else:
        prop = "P436"
        mbids = [g["mbid"] for g in gold]
    values = mb_values_sparql(mbids)
    return f"""
    SELECT DISTINCT ?item ?itemLabelEn ?itemLabelRu WHERE {{
      VALUES ?mbid {{ {values} }}
      ?item wdt:{prop} ?mbid .
      ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .
      OPTIONAL {{ ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") . }}
    }}
    """.strip()


def exact_ask_for_gold(answer_kind: str, gold: List[Dict[str, Any]]) -> str:
    if answer_kind == "performer":
        prop = "P434"
        mbids = [g["artist_mbid"] for g in gold]
    else:
        prop = "P436"
        mbids = [g["mbid"] for g in gold]
    values = mb_values_sparql(mbids)
    return f"""
    ASK WHERE {{
      BIND(wd:{{ITEM}} AS ?item)
      VALUES ?mbid {{ {values} }}
      ?item wdt:{prop} ?mbid .
    }}
    """.strip()


def nlg_release_query(kind: str, constraints: Dict[str, Any]) -> Tuple[str, str]:
    ru = f"Назови {constraints.get('requested_count', 5)} {kind_ru_acc(kind)}"
    en = f"Name {constraints.get('requested_count', 5)} {kind_en_plural(kind)}"
    parts_ru, parts_en = [], []
    if constraints.get("performer"):
        parts_ru.append(f"исполнителя «{constraints['performer_ru']}»")
        parts_en.append(f"by {constraints['performer']}")
    if constraints.get("genre"):
        parts_ru.append(f"в жанре «{genre_ru(constraints['genre'])}»")
        parts_en.append(f"in the {constraints['genre']} genre")
    y1, y2 = constraints.get("publication_year_min"), constraints.get("publication_year_max")
    if y1 is not None and y2 is not None:
        parts_ru.append(f"выпущенных в период {y1}–{y2} годов")
        parts_en.append(f"released between {y1} and {y2}")
    if constraints.get("evidence"):
        ev_ru = constraints.get("evidence_ru")
        ev_en = constraints.get("evidence_en")
        if ev_ru:
            parts_ru.append(ev_ru)
        if ev_en:
            parts_en.append(ev_en)
    if parts_ru:
        # No comma before performer immediately after object kind; commas before other modifiers are fine.
        if parts_ru[0].startswith("исполнителя "):
            ru += " " + parts_ru[0]
            rest = parts_ru[1:]
        else:
            rest = parts_ru
        if rest:
            ru += ", " + ", ".join(rest)
    if parts_en:
        en += " " + " ".join(parts_en)
    return ru + ".", en + "."


def evidence_phrase(cond: Dict[str, Any]) -> Tuple[str, str]:
    kind = cond["kind"]
    genre = cond.get("genre")
    y1, y2 = cond.get("year_min"), cond.get("year_max")
    ru = f"есть {release_singular_ru(kind)}"
    en = f"have a {release_singular_en(kind)}"
    if genre:
        ru += f" в жанре «{genre_ru(genre)}»"
        en += f" in the {genre} genre"
    if y1 is not None and y2 is not None:
        ru += f", выпущенный в период {y1}–{y2} годов"
        en += f" released between {y1} and {y2}"
    return ru, en


def nlg_performer_query(conditions: List[Dict[str, Any]], requested: int) -> Tuple[str, str]:
    parts_ru, parts_en = [], []
    for cond in conditions:
        ru, en = evidence_phrase(cond)
        parts_ru.append(ru)
        parts_en.append(en)
    ru = f"Назови {requested} исполнителей, у которых " + " и ".join(parts_ru) + "."
    en = f"Name {requested} performers who " + " and ".join(parts_en) + "."
    return ru, en


def make_record_from_gold(
    *,
    level: str,
    answer_kind: str,
    quota_bucket: str,
    template_id: str,
    constraints: Dict[str, Any],
    query_text_ru: str,
    query_text_en: str,
    gold: List[Dict[str, Any]],
    local_filter_meta: Dict[str, Any],
    evidence_by_answer: Optional[List[Dict[str, Any]]] = None,
) -> Optional[BenchmarkExample]:
    requested = REQUESTED_BY_LEVEL[level]
    ok, reason = candidate_ok(gold, level, requested)
    if not ok:
        return None
    qids = [g["qid"] for g in gold]
    labels_en = [g["label_en"] for g in gold]
    labels_ru = [g.get("label_ru") or g["label_en"] for g in gold]
    if len(qids) != len(set(qids)):
        return None
    if len(labels_en) != len(qids) or len(labels_ru) != len(qids):
        return None
    sparql_query = exact_sparql_for_gold(answer_kind, gold)
    ask_validator_sparql = exact_ask_for_gold(answer_kind, gold)

    constraints_clean = dict(constraints)
    constraints_clean.pop("requested_count", None)
    constraints_clean = {k: v for k, v in constraints_clean.items() if not k.endswith("_ru") and k not in {"evidence", "evidence_ru", "evidence_en"}}

    local_validator = {
        "type": "musicbrainz_linked_cache_exact",
        "source": "MusicBrainz API local cache + Wikidata MusicBrainz-ID mapping",
        "applies_after": "ask_validator_sparql",
        "filters": constraints_clean,
        "gold_mbid_set": [g.get("artist_mbid") if answer_kind == "performer" else g.get("mbid") for g in gold],
        "note": "Gold universe is MusicBrainz entities linked to Wikidata with English labels. Russian labels may fall back to original/English titles.",
    }

    meta = {
        "source": "musicbrainz_api_plus_wikidata_mbid_mapping",
        "generator_version": MUSIC_GENERATOR_VERSION,
        "coverage_policy": "musicbrainz_local_cache_wikidata_linked_gold_en_label_required",
        "gold_scope_policy": "Gold is exact within the locally built MusicBrainz-linked cache. v32 uses a broader cache and avoids query patterns whose real-world answer set is obviously broader than the cache can support.",
        "answer_kind": answer_kind,
        "answer_kind_quota_bucket": quota_bucket,
        "template_id": template_id,
        "template_family": "music_release_musicbrainz_backed",
        "constraints_are_clean_user_facing": True,
        "constraints_language": "en",
        "forbidden_properties": ["P495"],
        "country_constraints_used": False,
        "spotify_stream_count_used": False,
        "streams_policy": "Not used: MusicBrainz/Wikidata do not provide stable complete stream-count facts suitable for exact gold collection.",
        "chart_policy": "Not used: chart facts are sparse and unstable for exact gold collection.",
        "musicbrainz_cache_built_at": mb_cache_meta.get("built_at"),
        "musicbrainz_release_groups_in_cache": mb_cache_meta.get("linked_release_groups"),
        "musicbrainz_artists_in_cache": mb_cache_meta.get("linked_artists"),
        "local_gold_filter": local_filter_meta,
        "gold_returned": len(gold),
        "gold_total_before_limit": len(gold),
        "max_gold_allowed": MAX_GOLD_ALLOWED,
        # This is not a WDQS cached query result. Gold is built from a local
        # MusicBrainz cache, then mapped to Wikidata QIDs by MusicBrainz IDs.
        "gold_queries_used_cache": False,
        "musicbrainz_local_cache_used": True,
        "gold_query_mode": "local_musicbrainz_cache_select_first_not_live_wdqs_search",
        "english_label_required": True,
        "russian_label_policy": "RU label optional; fallback to original/English title is allowed and intentional for music releases.",
        "gold_may_be_incomplete_due_to_wdqs_limit": False,
        "gold_may_be_incomplete_due_to_count_mismatch": False,
        "gold_has_items_without_en_label": False,
        "gold_truncated_by_local_limit": False,
        "quality_filter_kind": "complete_local_musicbrainz_linked_gold_no_country_no_duplicate_public_labels",
        "duplicate_public_label_policy": "reject_normalized_duplicate_en_or_ru_labels",
        "constraints_signature": constraints_signature(constraints_clean),
        "min_gold_required": MIN_GOLD_BY_LEVEL[level],
    }
    if evidence_by_answer:
        meta["evidence_by_answer"] = evidence_by_answer

    return BenchmarkExample(
        id="__PENDING_ID__",
        domain=MUSIC_DOMAIN_NAME,
        complexity=level,
        query_text_ru=query_text_ru,
        constraints=constraints_clean,
        requested_count=requested,
        gold_answer_qids=qids,
        gold_answer_labels_ru=labels_ru,
        sparql_query=sparql_query,
        created_at=utc_now_z(),
        query_text_en=query_text_en,
        gold_answer_labels_en=labels_en,
        is_advanced=level in {"L4", "L5"},
        template_id=template_id,
        template_family="music_release_musicbrainz_backed",
        gold_truncated=False,
        ask_validator_sparql=ask_validator_sparql,
        local_validator=local_validator,
        gold_collection_meta=meta,
        gold_answer_imdb_ids=[],
        gold_answer_imdb_titles=[],
    )



## Candidate generation from local cache

In [ ]:


def all_known_genres(df: pd.DataFrame, min_count: int = 5) -> List[str]:
    c = Counter()
    for row in df.itertuples(index=False):
        for g in row_genres(row):
            if g in GENRE_RU:
                c[g] += 1
    return [g for g, n in c.most_common() if n >= min_count]

KNOWN_GENRES = all_known_genres(release_df, min_count=4)
print("known genres in linked cache:", len(KNOWN_GENRES), KNOWN_GENRES[:30])


def build_release_candidate(level: str, kind: str, template_id: str, *, genre: Optional[str] = None, performer: Optional[str] = None, performer_ru: Optional[str] = None, year_min: Optional[int] = None, year_max: Optional[int] = None, base_df: Optional[pd.DataFrame] = None, evidence_conditions: Optional[List[Dict[str, Any]]] = None) -> Optional[BenchmarkExample]:
    requested = REQUESTED_BY_LEVEL[level]
    df = filter_releases(kind=kind, genre=genre, performer=performer, year_min=year_min, year_max=year_max, base_df=base_df)
    evidence_by_answer = []
    # Optional bridge: keep only releases whose performer has all evidence conditions.
    if evidence_conditions:
        allowed_artists: Optional[Set[str]] = None
        evidence_lookup: Dict[Tuple[str, str], Dict[str, Any]] = {}
        for idx, cond in enumerate(evidence_conditions):
            evdf = filter_releases(kind=cond.get("kind"), genre=cond.get("genre"), year_min=cond.get("year_min"), year_max=cond.get("year_max"))
            artists = set(evdf["main_artist_mbid"].dropna().astype(str).tolist()) if len(evdf) else set()
            allowed_artists = artists if allowed_artists is None else allowed_artists.intersection(artists)
            for row in evdf.sort_values(["main_artist_mbid", "first_year", "label_en"]).itertuples(index=False):
                key = (str(row.main_artist_mbid), f"ev{idx}")
                if key not in evidence_lookup:
                    evidence_lookup[key] = {
                        "condition_index": idx,
                        "evidence_release_qid": row.qid,
                        "evidence_release_mbid": row.mbid,
                        "evidence_release_title_en": row.label_en,
                        "evidence_release_title_ru": row.label_ru or row.label_en,
                        "evidence_release_kind": row.kind,
                        "evidence_release_year": int(row.first_year),
                    }
        if allowed_artists is None:
            allowed_artists = set()
        df = df[df["main_artist_mbid"].isin(allowed_artists)] if len(df) else df
        for row in df.sort_values(["qid"]).itertuples(index=False):
            evs = [evidence_lookup.get((str(row.main_artist_mbid), f"ev{i}")) for i in range(len(evidence_conditions))]
            evidence_by_answer.append({
                "answer_qid": row.qid,
                "answer_mbid": row.mbid,
                "answer_label_en": row.label_en,
                "artist_qid": row.main_artist_qid,
                "artist_label_en": row.main_artist_label_en,
                "evidence": [e for e in evs if e],
            })
    gold = release_gold_from_df(df)
    constraints = {"kind": kind, "requested_count": requested}
    if performer:
        constraints["performer"] = performer
        constraints["performer_ru"] = performer_ru or performer
    if genre:
        constraints["genre"] = genre
    if year_min is not None and year_max is not None:
        constraints["publication_year_min"] = int(year_min)
        constraints["publication_year_max"] = int(year_max)
    if evidence_conditions:
        # Keep clean user-facing constraints flat. The NLG-only keys are removed later.
        constraints["evidence"] = evidence_conditions
        ev_ru_parts, ev_en_parts = [], []
        for j, cond in enumerate(evidence_conditions, start=1):
            constraints[f"performer_also_has_kind_{j}"] = cond.get("kind")
            if cond.get("genre"):
                constraints[f"performer_also_has_genre_{j}"] = cond.get("genre")
            if cond.get("year_min") is not None and cond.get("year_max") is not None:
                constraints[f"performer_also_has_publication_year_min_{j}"] = int(cond.get("year_min"))
                constraints[f"performer_also_has_publication_year_max_{j}"] = int(cond.get("year_max"))
            ru, en = evidence_phrase(cond)
            ev_ru_parts.append("у исполнителей, у которых также " + ru if not ev_ru_parts else "также " + ru)
            ev_en_parts.append("by performers who also " + en if not ev_en_parts else "also " + en)
        constraints["evidence_ru"] = " и ".join(ev_ru_parts)
        constraints["evidence_en"] = " and ".join(ev_en_parts)
    query_ru, query_en = nlg_release_query(kind, constraints)
    local_meta = {"answer_filter": {"kind": kind, "genre": genre, "performer": performer, "year_min": year_min, "year_max": year_max}, "evidence_conditions": evidence_conditions or []}
    return make_record_from_gold(
        level=level,
        answer_kind=kind,
        quota_bucket=release_quota_bucket(kind),
        template_id=template_id,
        constraints=constraints,
        query_text_ru=query_ru,
        query_text_en=query_en,
        gold=gold,
        local_filter_meta=local_meta,
        evidence_by_answer=evidence_by_answer or None,
    )


def build_performer_candidate(level: str, template_id: str, conditions: List[Dict[str, Any]]) -> Optional[BenchmarkExample]:
    requested = REQUESTED_BY_LEVEL[level]
    filtered_sets = []
    ev_dfs = []
    for cond in conditions:
        df = filter_releases(kind=cond.get("kind"), genre=cond.get("genre"), year_min=cond.get("year_min"), year_max=cond.get("year_max"))
        ev_dfs.append(df)
        filtered_sets.append(set(df["main_artist_mbid"].dropna().astype(str).tolist()) if len(df) else set())
    if not filtered_sets:
        return None
    allowed = set.intersection(*filtered_sets) if filtered_sets else set()
    if not allowed:
        return None
    # Build a performer evidence table from the first condition; then attach evidence for all conditions.
    all_rows = release_df[release_df["main_artist_mbid"].isin(allowed)].copy()
    gold = performer_gold_from_release_df(all_rows)
    # Enforce that answer performers have WD QIDs via performer_gold_from_release_df.
    evidence_by_answer = []
    for g in gold:
        evs = []
        for i, df in enumerate(ev_dfs):
            sub = df[df["main_artist_mbid"] == g["artist_mbid"]]
            if len(sub):
                row = sub.sort_values(["first_year", "label_en"]).iloc[0]
                evs.append({
                    "condition_index": i,
                    "evidence_release_qid": row["qid"],
                    "evidence_release_mbid": row["mbid"],
                    "evidence_release_title_en": row["label_en"],
                    "evidence_release_title_ru": row["label_ru"] or row["label_en"],
                    "evidence_release_kind": row["kind"],
                    "evidence_release_year": int(row["first_year"]),
                })
        evidence_by_answer.append({
            "answer_qid": g["qid"],
            "answer_mbid": g["artist_mbid"],
            "answer_label_en": g["label_en"],
            "evidence": evs,
        })
    constraints = {"kind": "performer"}
    for j, cond in enumerate(conditions, start=1):
        constraints[f"released_kind_{j}"] = cond.get("kind")
        if cond.get("genre"):
            constraints[f"released_genre_{j}"] = cond.get("genre")
        if cond.get("year_min") is not None and cond.get("year_max") is not None:
            constraints[f"released_publication_year_min_{j}"] = int(cond.get("year_min"))
            constraints[f"released_publication_year_max_{j}"] = int(cond.get("year_max"))
    query_ru, query_en = nlg_performer_query(conditions, requested)
    local_meta = {"performer_conditions": conditions}
    return make_record_from_gold(
        level=level,
        answer_kind="performer",
        quota_bucket="performer",
        template_id=template_id,
        constraints=constraints,
        query_text_ru=query_ru,
        query_text_en=query_en,
        gold=gold,
        local_filter_meta=local_meta,
        evidence_by_answer=evidence_by_answer,
    )


def condition_key(cond: Dict[str, Any]) -> Tuple:
    return (cond.get("kind"), norm_text(cond.get("genre") or ""), cond.get("year_min"), cond.get("year_max"))


def condition_label(cond: Dict[str, Any]) -> str:
    return f"{cond.get('kind')}:{cond.get('genre') or '*'}:{cond.get('year_min') or '*'}-{cond.get('year_max') or '*'}"


def condition_size(cond: Dict[str, Any]) -> int:
    return len(filter_releases(kind=cond.get("kind"), genre=cond.get("genre"), year_min=cond.get("year_min"), year_max=cond.get("year_max")))


def performer_intersection_size(conditions: List[Dict[str, Any]]) -> int:
    sets = []
    for cond in conditions:
        df = filter_releases(kind=cond.get("kind"), genre=cond.get("genre"), year_min=cond.get("year_min"), year_max=cond.get("year_max"))
        sets.append(set(df["main_artist_mbid"].dropna().astype(str).tolist()) if len(df) else set())
    return len(set.intersection(*sets)) if sets else 0


def make_condition_bank() -> List[Dict[str, Any]]:
    """Local condition bank for bridge/evidence tasks.

    Conditions intentionally avoid performer names. This keeps prompts from collapsing
    into a single artist's discography and makes L3-L5 genuinely multi-hop: answer
    releases/performers are selected through genre/year/kind evidence across an artist's
    career in the local MusicBrainz-linked cache.
    """
    out: List[Dict[str, Any]] = []
    seen: Set[Tuple] = set()
    genres = KNOWN_GENRES[:32]
    # Use both 5-year and decade buckets. 5-year buckets give specificity; decade
    # buckets prevent intersections from vanishing at hard levels.
    periods = YEAR_BUCKETS_5 + YEAR_BUCKETS_10 + [(1960, 2021)]
    for kind in ["studio album", "EP", "single"]:
        for genre in genres:
            # genre-only conditions are useful as evidence bridges at L4/L5
            for y1, y2 in [(None, None)] + periods:
                cond = {"kind": kind, "genre": genre}
                if y1 is not None:
                    cond["year_min"] = y1
                    cond["year_max"] = y2
                key = condition_key(cond)
                if key in seen:
                    continue
                seen.add(key)
                n = condition_size(cond)
                if MIN_GOLD_BY_LEVEL["L5"] <= n <= MAX_GOLD_ALLOWED:
                    cond["_size"] = n
                    out.append(cond)
    # Prefer medium-sized conditions and mix kinds/genres deterministically.
    out.sort(key=lambda c: (abs(c.get("_size", 0) - 16), c.get("kind"), c.get("genre") or "", c.get("year_min") or 0))
    return out


def compatible_condition_set(conditions: List[Dict[str, Any]], *, min_kinds: int = 2, min_genres: int = 2) -> bool:
    kinds = {c.get("kind") for c in conditions if c.get("kind")}
    genres = {norm_text(c.get("genre")) for c in conditions if c.get("genre")}
    if len(kinds) < min_kinds:
        return False
    if len(genres) < min_genres:
        return False
    return True


def build_candidate_records() -> List[BenchmarkExample]:
    candidates: List[BenchmarkExample] = []
    seen_sigs: Set[Tuple[str, str, str]] = set()

    def add(ex: Optional[BenchmarkExample]):
        if ex is None:
            return
        sig = (ex.complexity, ex.template_id, constraints_signature(ex.constraints))
        if sig in seen_sigs:
            return
        seen_sigs.add(sig)
        candidates.append(ex)

    condition_bank = make_condition_bank()
    print("condition bank:", len(condition_bank), Counter(c["kind"] for c in condition_bank))
    print("condition bank sample:", [condition_label(c) for c in condition_bank[:12]])

    # ------------------------------------------------------------------
    # L1: simple but not performer-anchored. We deliberately avoid prompts
    # like "albums by The Beatles" because they are too easy/narrow.
    # ------------------------------------------------------------------
    for genre in KNOWN_GENRES[:32]:
        for y1, y2 in YEAR_BUCKETS_5 + YEAR_BUCKETS_10:
            add(build_release_candidate("L1", "studio album", "ma_mb_album_l1_genre_period", genre=genre, year_min=y1, year_max=y2))
            add(build_release_candidate("L1", "EP", "ma_mb_EP_l1_genre_period", genre=genre, year_min=y1, year_max=y2))

    # ------------------------------------------------------------------
    # L2: release answers with genre/year and light evidence bridges. No
    # direct performer constraint. Singles are allowed, but capped by target.
    # ------------------------------------------------------------------
    base_genres = KNOWN_GENRES[:30]
    for genre in base_genres:
        for y1, y2 in YEAR_BUCKETS_5 + YEAR_BUCKETS_10:
            for kind in ["studio album", "EP", "single"]:
                add(build_release_candidate("L2", kind, f"ma_mb_{release_quota_bucket(kind)}_l2_genre_period", genre=genre, year_min=y1, year_max=y2))

    # L2 bridge: answer releases in genre/period whose performers also have another release kind.
    # This is more interesting than performer=X, but not as hard as L3/L4 evidence intersections.
    for ans_kind in ["studio album", "EP", "single"]:
        for genre in base_genres[:22]:
            for y1, y2 in YEAR_BUCKETS_10:
                for ev_kind in [k for k in ["studio album", "EP", "single"] if k != ans_kind]:
                    ev = {"kind": ev_kind, "genre": genre}
                    add(build_release_candidate(
                        "L2", ans_kind,
                        f"ma_mb_{release_quota_bucket(ans_kind)}_l2_genre_period_artist_also_has_{release_quota_bucket(ev_kind)}_same_genre",
                        genre=genre, year_min=y1, year_max=y2, evidence_conditions=[ev]
                    ))

    # ------------------------------------------------------------------
    # L3: one non-trivial evidence bridge for release answers, or two evidence
    # conditions for performer answers.
    # ------------------------------------------------------------------
    mid_conditions = condition_bank[:220]
    for ans_kind in ["studio album", "EP", "single"]:
        for genre in base_genres[:24]:
            for y1, y2 in YEAR_BUCKETS_10 + [(1960, 2021)]:
                for ev in mid_conditions[:90]:
                    if ev.get("kind") == ans_kind and norm_text(ev.get("genre")) == norm_text(genre):
                        continue
                    add(build_release_candidate(
                        "L3", ans_kind,
                        f"ma_mb_{release_quota_bucket(ans_kind)}_l3_genre_period_artist_also_has_one_release_condition",
                        genre=genre, year_min=y1, year_max=y2,
                        evidence_conditions=[{k: v for k, v in ev.items() if not k.startswith("_")}]
                    ))
    # Performer answers with two release conditions.
    for i, c1 in enumerate(mid_conditions[:160]):
        for c2 in mid_conditions[i+1:i+30]:
            pair = [c1, c2]
            if not compatible_condition_set(pair, min_kinds=2, min_genres=1):
                continue
            if performer_intersection_size(pair) > MAX_GOLD_ALLOWED:
                continue
            add(build_performer_candidate(
                "L3",
                "ma_mb_performer_l3_two_release_conditions",
                [{k: v for k, v in c.items() if not k.startswith("_")} for c in pair]
            ))

    # ------------------------------------------------------------------
    # L4: release answers with two evidence conditions and performer answers
    # with two/three more specific conditions.
    # ------------------------------------------------------------------
    for ans_kind in ["studio album", "EP", "single"]:
        for genre in base_genres[:24]:
            for y1, y2 in YEAR_BUCKETS_10 + [(1960, 2021)]:
                ev_pool = [c for c in mid_conditions[:180] if c.get("kind") != ans_kind and norm_text(c.get("genre")) != norm_text(genre)]
                for a in range(0, min(len(ev_pool), 80), 7):
                    for b in range(a + 1, min(len(ev_pool), a + 28), 9):
                        evs = [ev_pool[a], ev_pool[b]]
                        if not compatible_condition_set(evs + [{"kind": ans_kind, "genre": genre}], min_kinds=2, min_genres=2):
                            continue
                        add(build_release_candidate(
                            "L4", ans_kind,
                            f"ma_mb_{release_quota_bucket(ans_kind)}_l4_genre_period_artist_also_has_two_release_conditions",
                            genre=genre, year_min=y1, year_max=y2,
                            evidence_conditions=[{k: v for k, v in e.items() if not k.startswith("_")} for e in evs]
                        ))
    # Performer answer: two medium conditions, often with year windows.
    for i, c1 in enumerate(mid_conditions[:180]):
        for c2 in mid_conditions[i+1:i+36]:
            pair = [c1, c2]
            if not compatible_condition_set(pair, min_kinds=2, min_genres=2):
                continue
            if performer_intersection_size(pair) > MAX_GOLD_ALLOWED:
                continue
            add(build_performer_candidate(
                "L4", "ma_mb_performer_l4_two_specific_release_conditions",
                [{k: v for k, v in c.items() if not k.startswith("_")} for c in pair]
            ))

    # ------------------------------------------------------------------
    # L5: hard multi-hop. Release answers require answer filters plus three
    # evidence conditions; performer answers require three evidence conditions.
    # ------------------------------------------------------------------
    hard_conditions = condition_bank[:260]
    for ans_kind in ["studio album", "EP", "single"]:
        for genre in base_genres[:20]:
            for y1, y2 in YEAR_BUCKETS_10 + [(1960, 2021)]:
                ev_pool = [c for c in hard_conditions if c.get("kind") != ans_kind or norm_text(c.get("genre")) != norm_text(genre)]
                for a in range(0, min(len(ev_pool), 90), 11):
                    for b in range(a + 1, min(len(ev_pool), a + 35), 11):
                        for cidx in range(b + 1, min(len(ev_pool), b + 30), 13):
                            evs = [ev_pool[a], ev_pool[b], ev_pool[cidx]]
                            if not compatible_condition_set(evs + [{"kind": ans_kind, "genre": genre}], min_kinds=2, min_genres=3):
                                continue
                            add(build_release_candidate(
                                "L5", ans_kind,
                                f"ma_mb_{release_quota_bucket(ans_kind)}_l5_genre_period_artist_also_has_three_release_conditions",
                                genre=genre, year_min=y1, year_max=y2,
                                evidence_conditions=[{k: v for k, v in e.items() if not k.startswith("_")} for e in evs]
                            ))
    # Performer answer with three conditions. Use broad and specific conditions mixed.
    for i, c1 in enumerate(hard_conditions[:220]):
        for j, c2 in enumerate(hard_conditions[i+1:i+34], start=i+1):
            for c3 in hard_conditions[j+1:j+22]:
                triple = [c1, c2, c3]
                if not compatible_condition_set(triple, min_kinds=2, min_genres=2):
                    continue
                n = performer_intersection_size(triple)
                if n > MAX_GOLD_ALLOWED:
                    continue
                add(build_performer_candidate(
                    "L5", "ma_mb_performer_l5_three_release_conditions",
                    [{k: v for k, v in c.items() if not k.startswith("_")} for c in triple]
                ))

    print("candidate records:", len(candidates))
    print("by level:", dict(Counter(c.complexity for c in candidates)))
    print("by kind:", dict(Counter((c.gold_collection_meta or {}).get("answer_kind_quota_bucket") for c in candidates)))
    print("by template top:", Counter(c.template_id for c in candidates).most_common(30))
    return candidates

candidate_records = build_candidate_records()



## Deterministic selection with diversity caps and output writing

In [ ]:
def evidence_condition_count(ex: BenchmarkExample) -> int:
    meta = ex.gold_collection_meta or {}
    ev = meta.get("local_gold_filter", {}).get("evidence_conditions") or meta.get("local_gold_filter", {}).get("performer_conditions") or []
    return len(ev) if isinstance(ev, list) else 0


def candidate_sort_key(ex: BenchmarkExample) -> Tuple:
    meta = ex.gold_collection_meta or {}
    n = len(ex.gold_answer_qids)
    # Prefer hard/multihop candidates and medium-sized gold sets. The extra hash
    # stable-shuffles similar candidates so selected records don't appear in obvious pairs.
    preferred_gold = abs(n - 14)
    ev_count = evidence_condition_count(ex)
    sig = constraints_signature(ex.constraints)
    stable_mix = int(hashlib.md5((ex.complexity + (ex.template_id or "") + sig).encode("utf-8")).hexdigest()[:8], 16) % 997
    return (ex.complexity, -ev_count, preferred_gold, stable_mix, ex.template_id or "", sig)


def can_add_with_caps(ex: BenchmarkExample, selected: List[BenchmarkExample], caps: Dict[str, int]) -> bool:
    level = ex.complexity
    sig = constraints_signature(ex.constraints)
    if sum(1 for r in selected if constraints_signature(r.constraints) == sig) >= caps["same_constraint_signature"]:
        return False
    if sum(1 for r in selected if r.complexity == level and r.template_id == ex.template_id) >= caps["same_template_per_level"]:
        return False
    artist = (ex.constraints or {}).get("performer")
    if artist and sum(1 for r in selected if (r.constraints or {}).get("performer") == artist) >= caps["same_artist_total"]:
        return False
    genre = (ex.constraints or {}).get("genre")
    if genre and sum(1 for r in selected if r.complexity == level and (r.constraints or {}).get("genre") == genre) >= caps["same_genre_per_level"]:
        return False
    # Avoid adjacent-looking pairs: same level + same template + same visible genre too often.
    if len(selected) >= 2:
        recent = selected[-2:]
        same_recent = sum(1 for r in recent if r.complexity == level and r.template_id == ex.template_id and (r.constraints or {}).get("genre") == genre)
        if same_recent >= 1 and caps["same_template_per_level"] <= 4:
            return False
    return True


def select_records(candidates: List[BenchmarkExample]) -> Tuple[List[BenchmarkExample], Dict[str, Any]]:
    """Select exactly TARGET_PLAN_MUSIC records if the candidate bank supports it.

    v30 bug: selection filled L1-L4 first and exhausted global kind quotas before
    L5 got a chance. v31 treats the per-level plan as primary and fills every
    level-kind bucket before any overflow. Global 25/25/25/25 kind balance is a
    target of the level-kind plan, not a reason to starve later levels.
    """
    levels = ["L1", "L2", "L3", "L4", "L5"]
    candidates = sorted(candidates, key=candidate_sort_key)
    selected: List[BenchmarkExample] = []
    selected_keys: Set[Tuple[str, str, str]] = set()
    diagnostics: Dict[str, Any] = {"passes": [], "missing_level_kind": []}

    def bucket_of(ex: BenchmarkExample) -> str:
        return (ex.gold_collection_meta or {}).get("answer_kind_quota_bucket")

    def key_of(ex: BenchmarkExample) -> Tuple[str, str, str]:
        return (ex.complexity, ex.template_id or "", constraints_signature(ex.constraints))

    def level_count(level: str) -> int:
        return sum(1 for r in selected if r.complexity == level)

    def level_kind_count(level: str, bucket: str) -> int:
        return sum(1 for r in selected if r.complexity == level and bucket_of(r) == bucket)

    def add_one(ex: BenchmarkExample) -> bool:
        k = key_of(ex)
        if k in selected_keys:
            return False
        selected.append(ex)
        selected_keys.add(k)
        return True

    def try_fill_pool(level: str, bucket: Optional[str], need: int, caps: Optional[Dict[str, int]]) -> int:
        if need <= 0:
            return 0
        pool = [c for c in candidates if c.complexity == level and (bucket is None or bucket_of(c) == bucket)]
        added = 0
        for ex in pool:
            if added >= need:
                break
            if key_of(ex) in selected_keys:
                continue
            if level_count(level) >= TARGET_PLAN_MUSIC[level]:
                break
            if caps is not None and not can_add_with_caps(ex, selected, caps):
                continue
            if add_one(ex):
                added += 1
        return added

    # Pass 1: fill the intended per-level kind plan. Try strict caps, then relaxed
    # caps, then no caps except duplicate signature. This prevents L5 starvation.
    for cap_name, caps in [("strict", DIVERSITY_CAPS_STRICT), ("relaxed", DIVERSITY_CAPS_RELAXED), ("no_caps", None)]:
        before = len(selected)
        for level in levels:
            for bucket, target in MUSIC_LEVEL_KIND_PLAN.get(level, {}).items():
                need = target - level_kind_count(level, bucket)
                if need > 0:
                    try_fill_pool(level, bucket, need, caps)
        diagnostics["passes"].append({"pass": f"planned_level_kind_{cap_name}", "added": len(selected) - before})

    # Record missing planned buckets, if any.
    for level in levels:
        for bucket, target in MUSIC_LEVEL_KIND_PLAN.get(level, {}).items():
            have = level_kind_count(level, bucket)
            if have < target:
                diagnostics["missing_level_kind"].append({"level": level, "kind": bucket, "target": target, "have": have})

    # Pass 2: fill any remaining level deficits, but only after every level-kind
    # bucket had a chance. This may slightly bend global kind balance if the cache
    # lacks some bucket, but it should still produce L5 instead of stopping at 75.
    for cap_name, caps in [("strict", DIVERSITY_CAPS_STRICT), ("relaxed", DIVERSITY_CAPS_RELAXED), ("no_caps", None)]:
        before = len(selected)
        for level in levels:
            need = TARGET_PLAN_MUSIC[level] - level_count(level)
            if need > 0:
                try_fill_pool(level, None, need, caps)
        diagnostics["passes"].append({"pass": f"level_deficit_fill_{cap_name}", "added": len(selected) - before})

    # Stable ordering and IDs.
    ordered: List[BenchmarkExample] = []
    idx = 1
    for level in levels:
        level_rows = [r for r in selected if r.complexity == level][:TARGET_PLAN_MUSIC[level]]
        for ex in level_rows:
            ex.id = f"music_albums_{level.lower()}_{idx:04d}"
            ordered.append(ex)
            idx += 1
    selected = ordered

    diagnostics.update({
        "selected": len(selected),
        "target_total": TARGET_TOTAL_MUSIC,
        "by_level": dict(Counter(r.complexity for r in selected)),
        "by_kind": dict(Counter(bucket_of(r) for r in selected)),
        "by_template": dict(Counter(r.template_id for r in selected)),
        "candidate_by_level": dict(Counter(c.complexity for c in candidates)),
        "candidate_by_kind": dict(Counter(bucket_of(c) for c in candidates)),
        "candidate_by_template_top": Counter(c.template_id for c in candidates).most_common(30),
        "gold_count_min": min([len(r.gold_answer_qids) for r in selected], default=0),
        "gold_count_max": max([len(r.gold_answer_qids) for r in selected], default=0),
    })
    return selected, diagnostics


def validate_selected_records(records: List[BenchmarkExample]) -> Dict[str, Any]:
    problems = []
    expected_fields = list(BenchmarkExample.__dataclass_fields__.keys())
    by_level = Counter(r.complexity for r in records)
    for level, target in TARGET_PLAN_MUSIC.items():
        if by_level.get(level, 0) != target:
            problems.append({"issue": "level_target_mismatch", "level": level, "target": target, "have": by_level.get(level, 0)})
    if len(records) != TARGET_TOTAL_MUSIC:
        problems.append({"issue": "total_target_mismatch", "target": TARGET_TOTAL_MUSIC, "have": len(records)})
    by_kind = Counter((r.gold_collection_meta or {}).get("answer_kind_quota_bucket") for r in records)
    for bucket, target in MUSIC_KIND_TARGET_PLAN.items():
        if by_kind.get(bucket, 0) != target:
            problems.append({"issue": "kind_target_mismatch", "kind": bucket, "target": target, "have": by_kind.get(bucket, 0)})

    for r in records:
        d = asdict(r)
        if list(d.keys()) != expected_fields:
            problems.append({"id": r.id, "issue": "schema_key_order_or_fields_mismatch"})
        if len(r.gold_answer_qids) != len(r.gold_answer_labels_en) or len(r.gold_answer_qids) != len(r.gold_answer_labels_ru):
            problems.append({"id": r.id, "issue": "gold_lengths_mismatch"})
        if len(set(r.gold_answer_qids)) != len(r.gold_answer_qids):
            problems.append({"id": r.id, "issue": "duplicate_qid"})
        if has_bad_public_label_duplicates([{"label_en": e, "label_ru": ru, "qid": q} for q, e, ru in zip(r.gold_answer_qids, r.gold_answer_labels_en, r.gold_answer_labels_ru)]):
            problems.append({"id": r.id, "issue": "duplicate_public_label"})
        c = r.constraints or {}
        bad_keys = [k for k in c if "qid" in k.lower() or "sparql" in k.lower() or k.startswith("P")]
        if bad_keys:
            problems.append({"id": r.id, "issue": "technical_key_in_constraints", "keys": bad_keys})
        if "P495" in (r.sparql_query or "") or "P495" in (r.ask_validator_sparql or ""):
            problems.append({"id": r.id, "issue": "country_property_P495_leaked"})
        meta = r.gold_collection_meta or {}
        if meta.get("gold_has_items_without_en_label"):
            problems.append({"id": r.id, "issue": "missing_en_label_flag"})
    return {
        "records": len(records),
        "problems": problems,
        "by_level": dict(Counter(r.complexity for r in records)),
        "by_kind": dict(Counter((r.gold_collection_meta or {}).get("answer_kind_quota_bucket") for r in records)),
        "by_template": dict(Counter(r.template_id for r in records)),
    }


def write_records(records: List[BenchmarkExample], diagnostics: Dict[str, Any]) -> None:
    validation = validate_selected_records(records)
    if validation["problems"] and not MUSIC_ALLOW_PARTIAL_OUTPUT:
        print("SELECTION DIAGNOSTICS:", json.dumps(diagnostics, ensure_ascii=False, indent=2)[:5000])
        print("VALIDATION PROBLEMS:", json.dumps(validation["problems"], ensure_ascii=False, indent=2)[:5000])
        raise RuntimeError(
            f"Refusing to write incomplete/invalid dataset: {len(records)}/{TARGET_TOTAL_MUSIC}. "
            "Set MUSIC_ALLOW_PARTIAL_OUTPUT=True only for debugging."
        )

    # Backup incompatible old output to avoid accidental resume/mix of versions.
    if MUSIC_OUT_PATH.exists():
        existing_version = None
        try:
            with MUSIC_OUT_PATH.open("r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        existing_version = (json.loads(line).get("gold_collection_meta") or {}).get("generator_version")
                        break
        except Exception:
            existing_version = None
        if existing_version != MUSIC_GENERATOR_VERSION:
            backup = MUSIC_OUT_PATH.with_suffix(f".backup_before_{MUSIC_GENERATOR_VERSION}.jsonl")
            if not backup.exists():
                MUSIC_OUT_PATH.rename(backup)
                print("backed up incompatible old output ->", backup)
    rows = [asdict(r) for r in records]
    _write_jsonl(MUSIC_OUT_PATH, rows)
    audit = {
        "generator_version": MUSIC_GENERATOR_VERSION,
        "created_at": utc_now_z(),
        "output_path": str(MUSIC_OUT_PATH),
        "target_plan": TARGET_PLAN_MUSIC,
        "answer_kind_target_plan": MUSIC_KIND_TARGET_PLAN,
        "cache_meta": mb_cache_meta,
        "selection_diagnostics": diagnostics,
        "validation": validation,
    }
    _json_dump(MUSIC_AUDIT_PATH, audit)
    _json_dump(MUSIC_CHECKPOINT_PATH, {"generator_version": MUSIC_GENERATOR_VERSION, "written": len(records), "updated_at": utc_now_z()})
    print("wrote", len(records), "records ->", MUSIC_OUT_PATH)
    print("audit ->", MUSIC_AUDIT_PATH)
    print("by level:", audit["validation"]["by_level"])
    print("by kind:", audit["validation"]["by_kind"])
    print("template counts:", audit["validation"]["by_template"])
    if audit["validation"]["problems"]:
        print("VALIDATION PROBLEMS:")
        for p in audit["validation"]["problems"][:20]:
            print(p)
    else:
        print("validation: OK")

selected_records, selection_diagnostics = select_records(candidate_records)
print("selected:", len(selected_records), selection_diagnostics)
if len(selected_records) < TARGET_TOTAL_MUSIC:
    print(f"WARN: selected only {len(selected_records)}/{TARGET_TOTAL_MUSIC}.")
write_records(selected_records, selection_diagnostics)



## Inspect generated records

In [ ]:

# Quick inspection table.
if MUSIC_OUT_PATH.exists():
    records_preview = _read_jsonl(MUSIC_OUT_PATH)
    print("records:", len(records_preview))
    print("by complexity:", dict(Counter(r.get("complexity") for r in records_preview)))
    print("by answer kind:", dict(Counter((r.get("gold_collection_meta") or {}).get("answer_kind_quota_bucket") for r in records_preview)))
    print("top templates:", Counter(r.get("template_id") for r in records_preview).most_common(20))
    print("sample queries:")
    for r in records_preview[:12]:
        print(r["id"], r["complexity"], (r.get("gold_collection_meta") or {}).get("answer_kind"), len(r.get("gold_answer_qids") or []), "|", r["query_text_ru"])
